In [1]:
import sys
print(sys.executable)

E:\anconda\envs\jupyter_env\python.exe


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
%matplotlib inline

import seaborn as sns

#This will hide some unnecessary warning messages so our notebook looks clean
import warnings
warnings.filterwarnings('ignore')

In [4]:
data_url = "https://raw.githubusercontent.com/avikumart/Road-Traffic-Severity-Classification-Project/main/RTA%20Dataset.csv"
df = pd.read_csv(data_url)
df.head()

HTTPError: HTTP Error 429: Too Many Requests

In [5]:
print(df.shape)


NameError: name 'df' is not defined

In [6]:
df.columns


NameError: name 'df' is not defined

In [7]:
# This is our target column - the things  we want to predict
df['Accident_severity'].value_counts(normalize=True)

NameError: name 'df' is not defined

In [8]:
#Lets Draw a bar chart to visualize the target column
plt.figure(figsize=(10,5))
sns.countplot(x='Accident_severity',data=df,order=df['Accident_severity'].value_counts().index)
plt.title('Accident Severity Count in each Severity Category')
plt.xlabel('Accident Severity')
plt.ylabel("Number of Accidents")

plt.show()

NameError: name 'df' is not defined

<Figure size 1000x500 with 0 Axes>

In [9]:
# Lets check how many missing(empty) values are present in each column
df.isnull().sum()

NameError: name 'df' is not defined

In [10]:
from pandas._libs import missing
# we only show columns that actually have missing values,sorted from highest to lowest
missing_values = df.isnull().sum()

data = missing_values[missing_values > 0].sort_values(ascending=False)
print(data)

NameError: name 'df' is not defined

In [11]:
# First, lets separate column names into 2 lists: categorical and numerical

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()

print("Categorical Columns:",categorical_cols,"\n\n")
print("\nNumerical Columns:",numerical_cols)


NameError: name 'df' is not defined

In [12]:
# Fill missing values in Categorical columns 
for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

  # Fill missing values in NUMERICAL columns with the median values of that column
for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

# Lets's double check - there should be zero missing values now
print("Total missing values remaining:\n",df.isnull().sum())


NameError: name 'categorical_cols' is not defined

In [13]:
# Let's look at the atime column first
df['Time'].head()

NameError: name 'df' is not defined

In [14]:
# pd.to_datetime() converts the text into a proper time format that python understands
#.dt.hour extracts only the hour part (a number from 0 to 23)

df['Hour']= pd.to_datetime(df['Time'],format = '%H:%M:%S').dt.hour

# now that we have extracted the hour , we dont need the original time column anymore
df = df.drop(columns=['Time'])

#lets check our new column
df[['Hour']].head()

NameError: name 'df' is not defined

In [15]:
df['Hour'].info()

NameError: name 'df' is not defined

In [16]:
df.head()

NameError: name 'df' is not defined

In [17]:
# these are the columns we will use as input features (things we know about the  accident scene)

feature_cols = [
    'Day_of_week',
    'Age_band_of_driver',
    'Sex_of_driver',
    'Educational_level',
    'Driving_experience',
    'Type_of_vehicle',
    'Area_accident_occured',
    'Lanes_or_Medians',
    'Road_allignment',
    'Types_of_Junction',
    'Road_surface_type',
    'Road_surface_conditions',
    'Light_conditions',
    'Weather_conditions',
    'Type_of_collision',
    'Number_of_vehicles_involved',
    'Number_of_casualties',
    'Vehicle_movement',
    'Cause_of_accident',
    'Hour'
]

# this is our target column (what we want to predict)
target_col ='Accident_severity'

# lets create a smaller, cleaner dataframe with only the columns we need 
df_model =  df[feature_cols + [target_col]].copy()

df_model.head()

NameError: name 'df' is not defined

### Work of Label-Encoder 
1. extract unique values
2. 2. sort the values
   3. 3. assign no to the vlues
      4. 4. replace


In [18]:
# Import the LabelEncoder tool from scikit-Learn
from sklearn.preprocessing import LabelEncoder

#we will create one encoder for Each categorical column, and store it in a dictionary
# so that we can remember how each column was encoded (useful if we want o decode later)
label_encoder = {}

# Find which columns (in our smaller df_model) are still text (categorical)
categorical_features = df_model.select_dtypes(include='object').columns.tolist()
print(categorical_features)


NameError: name 'df_model' is not defined

In [ ]:
# Remove the target column from this list - we will encode it separately
categorical_features.remove(target_col)

print("Categorical feature columns to encode:",categorical_features)

In [ ]:
# now lets encode each categorical feature column, one by one

for col in categorical_features:
  le = LabelEncoder()                           # create a new encoder
  df_model[col] = le.fit_transform(df_model[col]) # convert text to numbers
  label_encoder[col] = le                         # save the encoder for this column

df_model.head()

In [ ]:
label_encoder

In [ ]:
# Now lets encode the target column (Accident_serevity) separately
#we use a different encoder so we can easily convert numbers back to labels later

target_encoder = LabelEncoder()
df_model[target_col] = target_encoder.fit_transform(df_model[target_col])

for i, class_name in enumerate (target_encoder.classes_):
  print(f"{i}    ->   {class_name}")

In [ ]:
# X contains all our feature columns (everything except the target)
X = df_model[feature_cols]

# y contains only the target column (what we want to predict )
y = df_model[target_col]

print("Shape of X (feature):",X.shape)
print("Shape of y (target):",y.shape)

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = y)

print("Training set size:",X_train.shape[0], "rows")
print("Testing set size:",X_test.shape[0], "rows")


NameError: name 'X' is not defined

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

#we create a dictionary to store 3 models, so we can train

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced',max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced',max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced',n_estimators=100, max_depth=10, random_state=42)
}
print("Models created:", list(models.keys()))

In [ ]:
# Now Let's TRAIN each model using our training data
# .fit() is the command that actually teaches the model using X_train and y_train

trained_models = {}

for name, model in models.items():
  print(f"Training {name}...")
  model.fit(X_train, y_train)
  trained_models[name] = model

  print("\n all models trained successfully!")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#we will store the accuracy of each model here, to compare them at the end
accuracy = {}

for name, model in trained_models.items():
  # use the trained model to predict severity on the TEST data
  y_pred = model.predict(X_test)

  # Calculate accuracy: how many predictions were correct out of
  acc = accuracy_score(y_test, y_pred)
  accuracy[name] = acc

  print("="*60)
  print(f"MODEL: {name}")
  print("="*60)
  print(f"Accuracy: {acc:.2%}")
  print(f"\nClassification Report:\n {classification_report(y_test, y_pred)}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report

# Prepare data for plotting
metrics_data = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)

    # Iterate through each class (0, 1, 2) which map to Fatal, Serious, Slight injury
    for i, class_label_encoded in enumerate(target_encoder.classes_):
        if str(i) in report:
            metrics_data.append({
                'Model': name,
                'Class': class_label_encoded, # Use decoded class name
                'Metric': 'Precision',
                'Value': report[str(i)]['precision']
            })
            metrics_data.append({
                'Model': name,
                'Class': class_label_encoded,
                'Metric': 'Recall',
                'Value': report[str(i)]['recall']
            })
            metrics_data.append({
                'Model': name,
                'Class': class_label_encoded,
                'Metric': 'F1-Score',
                'Value': report[str(i)]['f1-score']
            })

metrics_df = pd.DataFrame(metrics_data)

g = sns.catplot(data=metrics_df, x='Class', y='Value', hue='Metric', col='Model', col_wrap=3, kind='bar', palette='viridis', height=5, aspect=1.2)
g.fig.suptitle('Comparison of Model Performance Metrics by Accident Severity Class', y=1.02, fontsize=16)
g.set_axis_labels("Accident Severity Class", "Value")
g.set_xticklabels(rotation=45, ha="right")
g.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()